<a href="https://colab.research.google.com/github/memyselfandglitch/TransformerFromScratch/blob/main/GoogleColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!pip install -q transformers accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
# If Colab has enough GPU memory, try:
# model_id = "Qwen/Qwen2.5-1.5B-Instruct"
# model_id = "Qwen/Qwen2.5-7B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)

model.eval()


device: cuda


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [18]:
print("model_type:", model.config.model_type)
print("layers:", model.config.num_hidden_layers)
print("attention heads:", model.config.num_attention_heads)
print("kv heads:", model.config.num_key_value_heads)
print("hidden size:", model.config.hidden_size)
print("attention implementation:", model.config._attn_implementation)


head_dim = model.config.hidden_size // model.config.num_attention_heads
print("head_dim:", head_dim)


model_type: qwen2
layers: 24
attention heads: 14
kv heads: 2
hidden size: 896
attention implementation: sdpa
head_dim: 64


In [19]:
prompt = "The red cat was very"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs, use_cache=True)

past = outputs.past_key_values

print(type(past))
print(type(past.layers[0]) if hasattr(past, "layers") else type(past[0]))


<class 'transformers.cache_utils.DynamicCache'>
<class 'transformers.cache_utils.DynamicLayer'>


In [20]:
import torch

def extract_kv_layers(past):
    kv_layers = []

    for i, item in enumerate(past):
        if not isinstance(item, tuple):
            raise TypeError(f"Layer {i} cache item is not a tuple: {type(item)}")

        if len(item) < 2:
            raise TypeError(f"Layer {i} cache tuple is too short: len={len(item)}")

        k = item[0]
        v = item[1]

        if not isinstance(k, torch.Tensor):
            raise TypeError(f"Layer {i} K is not a tensor: {type(k)}")

        if not isinstance(v, torch.Tensor):
            raise TypeError(f"Layer {i} V is not a tensor: {type(v)}")

        # Ignore any extra metadata after K and V.
        kv_layers.append((k, v))

    return kv_layers


def show_kv_cache(label, past):
    kv_layers = extract_kv_layers(past)

    k0, v0 = kv_layers[0]

    print(f"\n{label}")
    print("number of cached layers:", len(kv_layers))

    print("Layer 0 K shape:", tuple(k0.shape))
    print("Layer 0 V shape:", tuple(v0.shape))

    print("Layer 0 K stride:", k0.stride())
    print("Layer 0 V stride:", v0.stride())

    print("Layer 0 K contiguous:", k0.is_contiguous())
    print("Layer 0 V contiguous:", v0.is_contiguous())

    total_bytes = sum(
        k.numel() * k.element_size() + v.numel() * v.element_size()
        for k, v in kv_layers
    )

    print("Total KV cache:", f"{total_bytes / 1024**2:.2f} MiB")


In [21]:
show_kv_cache("After prompt prefill", past)



After prompt prefill
number of cached layers: 24
Layer 0 K shape: (1, 2, 5, 64)
Layer 0 V shape: (1, 2, 5, 64)
Layer 0 K stride: (640, 320, 64, 1)
Layer 0 V stride: (640, 320, 64, 1)
Layer 0 K contiguous: True
Layer 0 V contiguous: True
Total KV cache: 0.06 MiB


In [22]:
seq_lengths = [5, 128, 256, 512, 1024, 2048, 4096, 8192, 16384]

print(f"{'seq_len':>8} | {'KV cache MiB':>14} | {'Layer 0 K shape':>22}")
print("-" * 55)


for seq_len in seq_lengths:
    try:
        input_ids = torch.randint(
            low=100,
            high=min(1000, tokenizer.vocab_size),
            size=(1, seq_len),
            device=model.device,
        )

        with torch.no_grad():
            outputs = model(input_ids=input_ids, use_cache=True)

        past = outputs.past_key_values
        kv_layers = extract_kv_layers(past)
        k0, v0 = kv_layers[0]

        total_mib = sum(
            k.numel() * k.element_size() + v.numel() * v.element_size()
            for k, v in kv_layers
        ) / 1024**2

        print(f"{seq_len:8d} | {total_mib:14.2f} | {str(tuple(k0.shape)):>22}")

        del input_ids, outputs, past
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"{seq_len:8d} | FAILED: {e}")
        break


 seq_len |   KV cache MiB |        Layer 0 K shape
-------------------------------------------------------
       5 |           0.06 |          (1, 2, 5, 64)
     128 |           1.50 |        (1, 2, 128, 64)
     256 |           3.00 |        (1, 2, 256, 64)
     512 |           6.00 |        (1, 2, 512, 64)
    1024 |          12.00 |       (1, 2, 1024, 64)
    2048 |          24.00 |       (1, 2, 2048, 64)
    4096 |          48.00 |       (1, 2, 4096, 64)
    8192 |          96.00 |       (1, 2, 8192, 64)
   16384 | FAILED: CUDA out of memory. Tried to allocate 14.00 GiB. GPU 0 has a total capacity of 14.56 GiB of which 10.91 GiB is free. Including non-PyTorch memory, this process has 3.65 GiB memory in use. Of the allocated memory 3.33 GiB is allocated by PyTorch, and 198.70 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Manage

# Using flash attention to avoid heavy prefill attention computation

In [23]:
import gc
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("Flash SDP enabled:", torch.backends.cuda.flash_sdp_enabled())
print("Memory efficient SDP enabled:", torch.backends.cuda.mem_efficient_sdp_enabled())
print("Math SDP enabled:", torch.backends.cuda.math_sdp_enabled())


CUDA available: True
GPU: Tesla T4
Compute capability: (7, 5)
Flash SDP enabled: True
Memory efficient SDP enabled: True
Math SDP enabled: True


In [24]:
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def extract_kv_layers(past):
    kv_layers = []

    for item in past:
        # Newer HF DynamicCache may yield (K, V, extra_metadata)
        if isinstance(item, tuple) and len(item) >= 2:
            k, v = item[0], item[1]
            if isinstance(k, torch.Tensor) and isinstance(v, torch.Tensor):
                kv_layers.append((k, v))

    return kv_layers


def kv_cache_mib(past):
    kv_layers = extract_kv_layers(past)
    return sum(
        k.numel() * k.element_size() + v.numel() * v.element_size()
        for k, v in kv_layers
    ) / 1024**2


def load_qwen(attn_implementation):
    clear_gpu()

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        attn_implementation=attn_implementation,
    )

    model.eval()

    print("\nLoaded:", model_id)
    print("attention implementation:", attn_implementation)
    print("layers:", model.config.num_hidden_layers)
    print("attention heads:", model.config.num_attention_heads)
    print("kv heads:", model.config.num_key_value_heads)
    print("hidden size:", model.config.hidden_size)
    print("head dim:", model.config.hidden_size // model.config.num_attention_heads)

    return tokenizer, model


In [25]:
def benchmark_prefill(model, seq_lengths):
    print(f"{'seq_len':>8} | {'KV cache MiB':>14} | {'peak CUDA MiB':>14} | {'prefill ms':>10}")
    print("-" * 60)

    for seq_len in seq_lengths:
        try:
            clear_gpu()

            input_ids = torch.randint(
                low=100,
                high=model.config.vocab_size - 1,
                size=(1, seq_len),
                device=model.device,
            )

            torch.cuda.synchronize()
            start = time.perf_counter()

            with torch.no_grad():
                outputs = model(input_ids=input_ids, use_cache=True)

            torch.cuda.synchronize()
            elapsed_ms = (time.perf_counter() - start) * 1000

            kv_mib = kv_cache_mib(outputs.past_key_values)
            peak_mib = torch.cuda.max_memory_allocated() / 1024**2

            print(f"{seq_len:8d} | {kv_mib:14.2f} | {peak_mib:14.2f} | {elapsed_ms:10.2f}")

            del input_ids, outputs
            clear_gpu()

        except Exception as e:
            print(f"{seq_len:8d} | FAILED: {type(e).__name__}: {str(e)[:120]}")
            clear_gpu()
            break


In [26]:
seq_lengths = [512, 1024, 2048, 4096, 8192, 16384]

tokenizer, model = load_qwen("eager")
benchmark_prefill(model, seq_lengths)

del model
clear_gpu()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Loaded: Qwen/Qwen2.5-0.5B-Instruct
attention implementation: eager
layers: 24
attention heads: 14
kv heads: 2
hidden size: 896
head dim: 64
 seq_len |   KV cache MiB |  peak CUDA MiB | prefill ms
------------------------------------------------------------
     512 |           6.00 |        3090.54 |     184.85
    1024 |          12.00 |        3245.79 |     168.92
    2048 |          24.00 |        3556.80 |     421.79
    4096 |          48.00 |        4850.35 |    1026.67
    8192 |          96.00 |       10413.47 |    3320.27
   16384 | FAILED: OutOfMemoryError: CUDA out of memory. Tried to allocate 7.00 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.78 GiB is free. Inclu


In [27]:
tokenizer, model = load_qwen("sdpa")
benchmark_prefill(model, seq_lengths)

del model
clear_gpu()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Loaded: Qwen/Qwen2.5-0.5B-Instruct
attention implementation: sdpa
layers: 24
attention heads: 14
kv heads: 2
hidden size: 896
head dim: 64
 seq_len |   KV cache MiB |  peak CUDA MiB | prefill ms
------------------------------------------------------------
     512 |           6.00 |        3090.54 |      91.72
    1024 |          12.00 |        3245.79 |     188.92
    2048 |          24.00 |        3556.80 |     450.70
    4096 |          48.00 |        5152.46 |    1565.92
    8192 |          96.00 |       11529.58 |    6318.00
   16384 | FAILED: OutOfMemoryError: CUDA out of memory. Tried to allocate 14.00 GiB. GPU 0 has a total capacity of 14.56 GiB of which 10.07 GiB is free. Inc


In [28]:
from torch.nn.attention import SDPBackend, sdpa_kernel
with sdpa_kernel(SDPBackend.MATH):
    benchmark_prefill(model, [512, 1024, 2048, 4096, 8192, 16384])
with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
    benchmark_prefill(model, [512, 1024, 2048, 4096, 8192, 16384])


NameError: name 'model' is not defined